In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd
import argparse
import pathlib
import os

import numpy as np

import matplotlib.pyplot as plt

def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
phase_cut_date = "2023-06-01"
prev_phase_cut_date = "2023-06-01"
cutoff_value = 0.05 # cutoff value for ensemble member selection (fraction of best models)

## Plot predictions for cases/concentration for different cutoff dates

In [ ]:
objective = "cases_and_conc"
towns = ["Koblenz", "Kaiserslautern", "Mainz", "Ludwigshafen", "Trier"]
descriptions = towns

In [ ]:
# load pred
conc_data = {}
I_data = {}
for town in towns:
    conc_data[town] =  np.load(f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results/{phase_cut_date}_prev{prev_phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_log_concentration.npz")
    I_data[town] = np.load(f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results/{phase_cut_date}_prev{prev_phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_I7_reported.npz")

In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

import sys 

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from optimization import optimization_utils

# load data and base model
base_config = {
        # data selection settings
        "data_kwargs": {
            "town": town,
            "log_scale": True, # this only considers WW measurements, not case counts
        },

       "phase_cut_date": phase_cut_date, # date to split data into two phases
        "dt": 0.2,
        "T_max": 25, # dummy value

        "underreporting_model": "monotone_increasing"
}



In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=5, sharex=True, sharey="row", figsize=(13.6, 4), dpi=300, constrained_layout=True)

for i, town in enumerate(towns):
    base_config["data_kwargs"]["town"] = town
    base_config["phase_cut_date"] = phase_cut_date
    base_config["objective"] = objective

    hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_prev{prev_phase_cut_date}_{objective}/hparams.json"

    with open(hparams_path) as f:
        hparams = json.load(f)

    base_config.update(hparams)
    base_config["phase_cut_date"] = phase_cut_date
    data = optimization_utils.two_phase_integrative_model_load_data(base_config)

    quantiles = {q: jnp.quantile(conc_data[town]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

    conc_median = quantiles[0.5]
    conc_low  = quantiles[0.025]
    conc_high = quantiles[0.975]

    axs[0,i].scatter(data["conc_dates_train"], data["conc_train"], label="Training/validation\n(concentration)", color="#3d85c6ff",alpha=0.7, s=15)
    axs[0,i].scatter(data["conc_dates_val"], data["conc_val"], label=None, color="#3d85c6ff",alpha=0.7, s=15)
    # axs[0,i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")

    valid_idx = (data["t_all_idx"]*base_config["dt"] >= int(hparams.get("T_max")))
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.15, label="95% CI")
    conc_low  = quantiles[0.05]
    conc_high = quantiles[0.95]
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.3, label="90% CI")
    conc_low  = quantiles[0.25]
    conc_high = quantiles[0.75]
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.45, label="50% CI")
    axs[0,i].plot(data["dates_all"][valid_idx], conc_median, label="Median", color="#8B1000")
    axs[0,i].set_title(f"{descriptions[i]}")


    quantiles_I = {q: jnp.quantile(I_data[town]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}
    I7_median = quantiles_I[0.5]
    I7_low  = quantiles_I[0.025]
    I7_high = quantiles_I[0.975]

    axs[1,i].scatter(data["I_dates_train"], data["I_train"], label="Training/validation\n(reported cases)", color="goldenrod", alpha=0.7, s=15)
    axs[1,i].scatter(data["I_dates_val"], data["I_val"], label=None, color="goldenrod", alpha=0.7, s=15)
    axs[1,i].scatter(data["obs_dates_phase_2"], data["I_test"], label="Test", color="#595959", alpha=0.7, s=15)
    axs[1,i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")

    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.15, label="95% CI")
    I7_low  = quantiles_I[0.05]
    I7_high = quantiles_I[0.95]
    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.3, label="90% CI")
    I7_low  = quantiles_I[0.25]
    I7_high = quantiles_I[0.75]
    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.45, label="50% CI")
    axs[1,i].plot(data["dates_all"][1:], I7_median, label="Median", color="#8B1000")



axs[0,0].set_ylabel("Log fraction\nCOVID-19\nover PMMoV")
axs[1,0].set_ylabel("7-day moving \nsum of reported\ncases [#]")

#axs[0,1].set_yticklabels([])
#axs[0,2].set_yticklabels([])
#axs[1,1].set_yticklabels([])
#axs[1,2].set_yticklabels([])

axs[0,i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
axs[0,i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))

plt.tight_layout()
plt.savefig(f"sentisurv_fits.png", dpi=300, bbox_inches='tight')